In [ ]:
%matplotlib inline
import os
import sys
import json
import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt

sys.path.append('..')
from datasets.open_world_clad import OWCladDetection
from clad.detection.cladd import get_cladd_trainval, get_cladd_test

from args import ARGS

[INFO] No Detectron installation found, continuing without.


## Object Detection Dataset

In [8]:
root = '../../data' 
print(os.path.abspath(root))

train_sets, val_sets = get_cladd_trainval(root, avalanche=False)
test_sets = get_cladd_test(root, avalanche=False)

/workspace/data


# CLAD visualization with FiftyOne

In [4]:
import glob
import fiftyone as fo
import fiftyone.zoo as foz

In [10]:
# head 10
head = -1
train_images_patt = "/workspace/data/SSLAD-2D/labeled/train"
train_annotation_dir = "/workspace/data/SSLAD-2D/labeled/annotations/instance_train.json"

dataset = train_sets[0] # task 1 dataset

count = 0
# Create samples for your data
samples = []
for data in dataset:
    if count == head:
        break
    count += 1
    img_id = data[1]['image_id'].item()
    file_name = dataset.img_annotations[img_id]['file_name']
    print(f"Count: {count}")
    sample = fo.Sample(filepath=os.path.join(train_images_patt, file_name))

    # Convert detections to FiftyOne format
    
    width, height = data[1]["sizes"]

    detections = []
    for i, obj in enumerate(data[1]['boxes']):
        label = data[1]["labels"][i].item() # label

        # Bounding box coordinates should be relative values
        # in [0, 1] in the following format:
        # [top-left-x, top-left-y, width, height]
        voc2coco = lambda x: [x[0]/width, x[1]/height, x[2]/width, x[3]/height] # xmin, ymin, xmax, ymax
        bounding_box = voc2coco(obj) # xmin, ymin, xmax, ymax
        bounding_box[2] -= bounding_box[0] # xmax -> width: xmax - xmin
        bounding_box[3] -= bounding_box[1] # ymax -> height: ymax - ymin
 
        detections.append(
            fo.Detection(label=str(label), bounding_box=bounding_box)
        )

    # Store detections in a field name of your choice
    sample["ground_truth"] = fo.Detections(detections=detections)

    samples.append(sample)

Count: 1
Count: 2
Count: 3
Count: 4
Count: 5
Count: 6
Count: 7
Count: 8
Count: 9
Count: 10
Count: 11
Count: 12
Count: 13
Count: 14
Count: 15
Count: 16
Count: 17
Count: 18
Count: 19
Count: 20
Count: 21
Count: 22
Count: 23
Count: 24
Count: 25
Count: 26
Count: 27
Count: 28
Count: 29
Count: 30
Count: 31
Count: 32
Count: 33
Count: 34
Count: 35
Count: 36
Count: 37
Count: 38
Count: 39
Count: 40
Count: 41
Count: 42
Count: 43
Count: 44
Count: 45
Count: 46
Count: 47
Count: 48
Count: 49
Count: 50
Count: 51
Count: 52
Count: 53
Count: 54
Count: 55
Count: 56
Count: 57
Count: 58
Count: 59
Count: 60
Count: 61
Count: 62
Count: 63
Count: 64
Count: 65
Count: 66
Count: 67
Count: 68
Count: 69
Count: 70
Count: 71
Count: 72
Count: 73
Count: 74
Count: 75
Count: 76
Count: 77
Count: 78
Count: 79
Count: 80
Count: 81
Count: 82
Count: 83
Count: 84
Count: 85
Count: 86
Count: 87
Count: 88
Count: 89
Count: 90
Count: 91
Count: 92
Count: 93
Count: 94
Count: 95
Count: 96
Count: 97
Count: 98
Count: 99
Count: 100
Count: 1

In [ ]:
# Create dataset
dataset = fo.Dataset("clad-t1-train-dataset")
dataset.add_samples(samples)


In [ ]:
print('Before delte: {}'.format(fo.list_datasets()))
if 'clad-t1-train-dataset' in fo.list_datasets():
    fo.delete_dataset("clad-t1-train-dataset")
print('After delete: {}'.format(fo.list_datasets()))

In [ ]:
session = fo.launch_app(dataset)